# 1. Import libraries

In [1]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
import time
import math
import random
import json
import os
import re
from datetime import datetime
from pathlib import Path
from PIL import Image, ImageDraw

# 2. Parsing Collections

In [2]:
# url of site for parsing
url = "https://telegifter.ru/gifts/collections/"

In [3]:
# Check access
pages = requests.get(url) # connecting
soup = bs(pages.text, 'html.parser')
print(pages.status_code)

200


Nice, we can to parse this site!

In [4]:
# Result list for dataframe in future
result_list_collection = { 
    "collection_id": [],
    "collection_name": [],
    "collection_image_url": [],
    "collection_limit": [],
    "collection_base_price": []
}

In [5]:
collection_container = soup.find('div', class_='container-575')
collection_cards = collection_container.find_all('div', class_='mini-single')

print(f'📦 Found elements: {len(collection_cards)}')

📦 Found elements: 144


In [6]:
images_folder = 'images/collections'

if not os.path.exists(images_folder):
    os.makedirs(images_folder)
    print(f'✅ Folder created: {images_folder}')
else:
    print(f'📂 Folder is already exist: {images_folder}')

📂 Folder is already exist: images/collections


In [7]:
def download_image(url, folder, filename):
    try:
        # Get image
        img_data = requests.get(url).content
        
        # Determine file type
        if '.webp' in url:
            ext = '.webp'
        elif '.png' in url:
            ext = '.png'
        elif '.jpg' in url or '.jpeg' in url:
            ext = '.jpg'
        elif '.svg' in url:
            ext = '.svg'
        else:
            ext = '.webp' 
        
        # Full path
        filepath = os.path.join(folder, filename + ext)
        
        # Check exist
        if os.path.exists(filepath):
            print(f'⏭️ Skipped (Already exist): {filename}{ext}')
            return filepath
        
        # Save
        with open(filepath, 'wb') as handler:
            handler.write(img_data)
        
        print(f'✅ Downloaded: {filename}{ext}')
        return filepath
        
    except Exception as e:
        print(f'❌ Error {url}: {e}')
        return None

In [8]:
i = 0

%time
for card in collection_cards:
    present_title = card.find('div', class_='mini-single-title-gifts').text.strip()
    
    check = card.find('div', class_='mini-single-stars col').text.strip().split()[0]
    check2 = card.find('div', class_='mini-single-other col').text.strip().split()[1]
    
    if check == 'Pre-market' or check2 == '∞':
        print(f'{present_title} skip!')
        continue
    link = card.find('a')
    if not link or not link.get('href'):
        continue
    collection_url = link.get('href')

    collection_page = requests.get(collection_url)
    collection_soup = bs(collection_page.text, 'html.parser')

    time.sleep(1)

    try:
        present_count = int(collection_soup.find_all('div', class_='col-sm-6 col-xs-12 stat-table-info')[2].text.strip().replace(' ', ''))
    except:
        present_count = int(collection_soup.find_all('div', class_='col-sm-6 col-xs-12 stat-table-info')[3].text.strip().replace(' ', ''))
    
    try:
        present_price = math.ceil(float(collection_soup.find_all('div', class_='col-sm-6 col-xs-12 stat-table-info')[5].text.strip().split()[2]))
    except:
        try:
            present_price = math.ceil(float(collection_soup.find_all('div', class_='col-sm-6 col-xs-12 stat-table-info')[-1].text.strip().split()[2]))
        except:
            present_price = math.ceil(float(collection_soup.find_all('div', class_='col-sm-6 col-xs-12 stat-table-info')[6].text.strip().split()[2]))
            
            
    try:
        img_elem = collection_soup.find('div', class_='wp-block-image').find('img')
        if img_elem and img_elem.get('src'):
            img_url = img_elem.get('src')
            
            # Make full url
            if not img_url.startswith('http'):
                img_url = 'https://telegifter.ru' + img_url
            
            # Generate file name
            # Desk Calendar → desk_calendar
            filename = re.sub(r'[^a-zA-Z0-9]', '_', present_title).lower().strip('_')
            
            # Download to folder
            download_image(img_url, images_folder, filename)
        else:
            print(f'⚠️ No image for {present_title}')
            
    except Exception as e:
        print(f'⚠️ Error {present_title}: {e}')

    result_list_collection['collection_id'].append(i)
    result_list_collection['collection_name'].append(present_title)
    result_list_collection['collection_image_url'].append(filename)
    result_list_collection['collection_limit'].append(present_count)
    result_list_collection['collection_base_price'].append(present_price)
    
    print(f'{i} - {present_title} - {present_count} pieces - {present_price} TON')
    i += 1

CPU times: total: 0 ns
Wall time: 4.53 μs
1 May skip!
⏭️ Skipped (Already exist): artisan_brick.webp
0 - Artisan Brick - 7000 pieces - 20 TON
⏭️ Skipped (Already exist): astral_shard.webp
1 - Astral Shard - 10000 pieces - 2 TON
⏭️ Skipped (Already exist): b_day_candle.webp
2 - B-day Candle - 500000 pieces - 2 TON
⏭️ Skipped (Already exist): berry_box.webp
3 - Berry Box - 100000 pieces - 1 TON
⏭️ Skipped (Already exist): big_year.webp
4 - Big Year - 200000 pieces - 1 TON
⏭️ Skipped (Already exist): bling_binky.webp
5 - Bling Binky - 10000 pieces - 25 TON
⏭️ Skipped (Already exist): bonded_ring.webp
6 - Bonded Ring - 9000 pieces - 21 TON
Book skip!
⏭️ Skipped (Already exist): bow_tie.webp
7 - Bow Tie - 100000 pieces - 2 TON
⏭️ Skipped (Already exist): bunny_muffin.webp
8 - Bunny Muffin - 100000 pieces - 1 TON
Candle Lamp skip!
⏭️ Skipped (Already exist): candy_cane.webp
9 - Candy Cane - 600000 pieces - 1 TON
Case skip!
Christmas Tree skip!
⏭️ Skipped (Already exist): clover_pin.webp
10 -

# 3. Parsing symbols

In [9]:
# url of site for parsing
url = "https://telegifter.ru/gifts/rarest-symbol/"

In [ ]:
# Check access
pages = requests.get(url) 
soup = bs(pages.text, 'html.parser') 
print(pages.status_code)

200


In [11]:
# Result list for dataframe in future
result_list_symbol = { 
    "symbol_id": [],
    "symbol_name": [],
    "symbol_image_url": []
}

In [12]:
symbol_container = soup.find('div', class_='gifts-table-rare active container')
symbol_cards = symbol_container.find_all('div', class_='row gift-item')

print(f'📦 Found elements: {len(symbol_cards)}')

📦 Found elements: 718


In [13]:
images_folder = 'images/symbols'

if not os.path.exists(images_folder):
    os.makedirs(images_folder)
    print(f'✅ Folder created: {images_folder}')
else:
    print(f'📂 Folder is already exist: {images_folder}')

📂 Folder is already exist: images/symbols


In [14]:
%time
i = 0

for card in symbol_cards:
    symbol_title = card.find_all('div', class_='col-sm-3 col-xs-6')[1].text.strip()
    
    try:
        img_elem = card.find_all('div', class_='col-sm-3 col-xs-6')[0].find('img')
        if img_elem and img_elem.get('src'):
            img_url = img_elem.get('src')
            
            # Make full url
            if not img_url.startswith('http'):
                img_url = 'https://telegifter.ru' + img_url
            
            # Generate file name
            # Desk Calendar → desk_calendar
            filename = re.sub(r'[^a-zA-Z0-9]', '_', symbol_title).lower().strip('_')
            
            # Download to folder
            download_image(img_url, images_folder, filename)
            
            result_list_symbol['symbol_id'].append(i)
            result_list_symbol['symbol_name'].append(symbol_title)
            result_list_symbol['symbol_image_url'].append(filename)
            
            print(f"{i} - {symbol_title} added!")
            i += 1
        else:
            print(f'⚠️ No image for {symbol_title}')
            
    except Exception as e:
        print(f'⚠️ Error {symbol_title}: {e}')

CPU times: total: 0 ns
Wall time: 3.81 μs
⏭️ Skipped (Already exist): old_scull.webp
0 - Old Scull added!
⏭️ Skipped (Already exist): witch_hat.webp
1 - Witch Hat added!
⏭️ Skipped (Already exist): tarot_cards.webp
2 - Tarot Cards added!
⏭️ Skipped (Already exist): casket.webp
3 - Casket added!
⏭️ Skipped (Already exist): voodoo.webp
4 - Voodoo added!
⏭️ Skipped (Already exist): vampire_bat.webp
5 - Vampire Bat added!
⏭️ Skipped (Already exist): jack_o_lantern.webp
6 - Jack-O-Lantern added!
⏭️ Skipped (Already exist): tombstone.webp
7 - Tombstone added!
⏭️ Skipped (Already exist): mortar.webp
8 - Mortar added!
⏭️ Skipped (Already exist): hex_pot.webp
9 - Hex Pot added!
⏭️ Skipped (Already exist): creepy_eye.webp
10 - Creepy Eye added!
⏭️ Skipped (Already exist): magic_elixir.webp
11 - Magic Elixir added!
⏭️ Skipped (Already exist): zombie_hand.webp
12 - Zombie Hand added!
⏭️ Skipped (Already exist): haunted_house.webp
13 - Haunted House added!
⏭️ Skipped (Already exist): squeeze_bottle

# 4. Parsing backgrounds

In [15]:
# url of site for parsing
url = "https://telegifter.ru/gifts/rarest-background/"

In [ ]:
# Check access
pages = requests.get(url) 
soup = bs(pages.text, 'html.parser') 
print(pages.status_code)

200


In [17]:
# Result list for dataframe in future
result_list_bg = { 
    "background_id": [],
    "background_name": [],
    "background_image_url": []
}

In [18]:
bg_container = soup.find('div', class_='gifts-table-rare active container')
bg_cards = bg_container.find_all('div', class_='row gift-item')

print(f'📦 Found elements: {len(bg_cards)}')

📦 Found elements: 80


In [19]:
images_folder = 'images/bgs'

if not os.path.exists(images_folder):
    os.makedirs(images_folder)
    print(f'✅ Folder created: {images_folder}')
else:
    print(f'📂 Folder is already exist: {images_folder}')

📂 Folder is already exist: images/bgs


In [20]:
def extract_gradient_colors(gradient_style):
    """
    Returns: (color_start, color_end) or (None, None)
    """
    if not gradient_style:
        return None, None
    
    try:
        colors = re.findall(r'rgb\((\d+),\s*(\d+),\s*(\d+)\)', gradient_style)
        
        if len(colors) >= 2:
            # First color
            color_start = ','.join(colors[0])  # "54,55,56"
            # Last color
            color_end = ','.join(colors[-1])   # "14,15,15"
            return color_start, color_end
        elif len(colors) == 1:
            color_start = ','.join(colors[0])
            return color_start, color_start
        else:
            return None, None
    except:
        return None, None

In [21]:
def create_gradient_image(gradient_style, filename, folder, width=1000, height=1000):
    try:
        colors = re.findall(r'rgb\((\d+),\s*(\d+),\s*(\d+)\)', gradient_style)
        
        if len(colors) >= 2:
            color1 = tuple(int(c) for c in colors[0])
            color2 = tuple(int(c) for c in colors[-1])
            
            img = Image.new('RGB', (width, height))
            draw = ImageDraw.Draw(img)
            
            for y in range(height):
                r = int(color1[0] + (color2[0] - color1[0]) * y / height)
                g = int(color1[1] + (color2[1] - color1[1]) * y / height)
                b = int(color1[2] + (color2[2] - color1[2]) * y / height)
                draw.line((0, y, width, y), fill=(r, g, b))
            
            filepath = os.path.join(folder, filename + '.png')
            img.save(filepath, 'PNG')
            print(f'✅ Gradient created: {filename}.png')
            return filepath
        else:
            print(f'⚠️ Color failure {gradient_style}')
            return None
            
    except Exception as e:
        print(f'❌ Error: {e}')
        return None

In [22]:
i = 0

%time
for card in bg_cards:
    bg_title = card.find_all('div', class_='col-sm-3 col-xs-6')[1].text.strip()
    print(bg_title)
    
    try:
        rare_elem = card.find_all('div', class_='col-sm-3 col-xs-6')[0].find('div', class_='gifts-table-rare-bg')
        gradient_style = rare_elem.get('style').strip()

        colors = re.findall(r'rgb\((\d+),\s*(\d+),\s*(\d+)\)', gradient_style)

        if len(colors) >= 2:
            color_start = ','.join(colors[0])
            color_end = ','.join(colors[-1])
        elif len(colors) == 1:
            color_start = ','.join(colors[0])
            color_end = color_start
        else:
            color_start = ''
            color_end = ''

        filename = re.sub(r'[^a-zA-Z0-9]', '_', bg_title).lower().strip('_')

        gradient_path = create_gradient_image(gradient_style, filename, images_folder)

        result_list_bg['background_id'].append(i)
        result_list_bg['background_name'].append(bg_title)
        result_list_bg['background_image_url'].append(filename)

        print(f'{i} - {bg_title} added!')
        i += 1
            
    except Exception as e:
        print(f'⚠️ Error {bg_title}: {e}')

CPU times: total: 0 ns
Wall time: 4.05 μs
Black
✅ Gradient created: black.png
0 - Black added!
Onyx Black
✅ Gradient created: onyx_black.png
1 - Onyx Black added!
Gunmetal
✅ Gradient created: gunmetal.png
2 - Gunmetal added!
Marine Blue
✅ Gradient created: marine_blue.png
3 - Marine Blue added!
Mexican Pink
✅ Gradient created: mexican_pink.png
4 - Mexican Pink added!
Coral Red
✅ Gradient created: coral_red.png
5 - Coral Red added!
Fire Engine
✅ Gradient created: fire_engine.png
6 - Fire Engine added!
Dark Green
✅ Gradient created: dark_green.png
7 - Dark Green added!
Feldgrau
✅ Gradient created: feldgrau.png
8 - Feldgrau added!
Pacific Cyan
✅ Gradient created: pacific_cyan.png
9 - Pacific Cyan added!
Deep Cyan
✅ Gradient created: deep_cyan.png
10 - Deep Cyan added!
Mystic Pearl
✅ Gradient created: mystic_pearl.png
11 - Mystic Pearl added!
Strawberry
✅ Gradient created: strawberry.png
12 - Strawberry added!
Platinum
✅ Gradient created: platinum.png
13 - Platinum added!
Sapphire
✅ Gradie

# 5. Parsing Models

In [23]:
# url of site for parsing
url = "https://telegifter.ru/gifts/collections/"

In [ ]:
# Check access
pages = requests.get(url)
soup = bs(pages.text, 'html.parser') 
print(pages.status_code)

200


In [25]:
# Result list for dataframe in future
result_list_models = { 
    "model_id": [],
    "collection_id": [],
    "model_name": [],
    "model_image_url": []
}

In [26]:
collection_container = soup.find('div', class_='container-575')
collection_cards = collection_container.find_all('div', class_='mini-single')

print(f'📦 Found elements: {len(collection_cards)}')

📦 Found elements: 144


In [27]:
images_folder = 'images/models'

if not os.path.exists(images_folder):
    os.makedirs(images_folder)
    print(f'✅ Folder created: {images_folder}')
else:
    print(f'📂 Folder is already exist: {images_folder}')

📂 Folder is already exist: images/models


In [28]:
i = 0
y = 0

BASE_URL = 'https://telegifter.ru'
HEADERS = {'User-Agent': 'Mozilla/5.0'}

%time
for card in collection_cards:
    present_title = card.find('div', class_='mini-single-title-gifts').text.strip()

    check  = card.find('div', class_='mini-single-stars col').text.strip().split()[0]
    check2 = card.find('div', class_='mini-single-other col').text.strip().split()[1]

    if check == 'Pre-market' or check2 == '∞':
        print(f'{present_title} — skip!')
        continue

    link = card.find('a')
    if not link or not link.get('href'):
        continue

    collection_url = link.get('href')

    # find collection_id from result_list_collection
    col_id = None
    for idx, name in enumerate(result_list_collection['collection_name']):
        if name == present_title:
            col_id = result_list_collection['collection_id'][idx]
            break

    if col_id is None:
        print(f'⚠️ collection_id not found for {present_title}, skip')
        continue

    col_page = requests.get(collection_url, headers=HEADERS, timeout=15)
    col_soup = bs(col_page.text, 'html.parser')
    time.sleep(random.uniform(1.5, 3.0))

    script_tag = col_soup.find('script', {'id': 'models-data', 'type': 'application/json'})
    if not script_tag:
        print(f'⚠️ No models-data for {present_title}')
        y += 1
        continue

    models = json.loads(script_tag.string)
    col_slug = re.sub(r'[^a-zA-Z0-9]', '_', present_title).lower().strip('_')

    for model in models:
        model_name = model['name']
        img_path   = model['image_url']
        full_url   = BASE_URL + img_path

        model_slug = re.sub(r'[^a-zA-Z0-9]', '_', model_name).lower().strip('_')
        filename   = f'{col_slug}_{model_slug}'

        download_image(full_url, images_folder, filename)

        result_list_models['model_id'].append(i)
        result_list_models['collection_id'].append(col_id)
        result_list_models['model_name'].append(model_name)
        result_list_models['model_image_url'].append(filename)

        i += 1

    print(f'✅ {present_title}: {len(models)} models (collection_id={col_id})')
    y += 1

CPU times: total: 0 ns
Wall time: 3.81 μs
1 May — skip!
⏭️ Skipped (Already exist): artisan_brick_gold_bar.webp
⏭️ Skipped (Already exist): artisan_brick_gold_block.webp
⏭️ Skipped (Already exist): artisan_brick_usbrick.webp
⏭️ Skipped (Already exist): artisan_brick_cash_roll.webp
⏭️ Skipped (Already exist): artisan_brick_cherry_ruby.webp
⏭️ Skipped (Already exist): artisan_brick_diamond.webp
⏭️ Skipped (Already exist): artisan_brick_duck_bath.webp
⏭️ Skipped (Already exist): artisan_brick_el_dorado.webp
⏭️ Skipped (Already exist): artisan_brick_fifth_element.webp
⏭️ Skipped (Already exist): artisan_brick_gemstones.webp
⏭️ Skipped (Already exist): artisan_brick_jewelry_box.webp
⏭️ Skipped (Already exist): artisan_brick_oriental.webp
⏭️ Skipped (Already exist): artisan_brick_pearl.webp
⏭️ Skipped (Already exist): artisan_brick_treasure.webp
⏭️ Skipped (Already exist): artisan_brick_bomb_planted.webp
⏭️ Skipped (Already exist): artisan_brick_brick_3310.webp
⏭️ Skipped (Already exist): ar

# Analisys

In [15]:
df_collections = pd.read_csv('../docs/collections.csv')
df_symbols     = pd.read_csv('../docs/symbols.csv')
df_bgs         = pd.read_csv('../docs/backgrounds.csv')
df_models      = pd.read_csv('../docs/models.csv')

In [16]:
print('*'*35)
print(f'Collections : {len(df_collections)}')
print('-'*35)
print(f'Models      : {len(df_models)}')
print('-'*35)
print(f'Symbols     : {len(df_symbols)}')
print('-'*35)
print(f'Backgrounds : {len(df_bgs)}')
print('*'*35)

***********************************
Collections : 111
-----------------------------------
Models      : 7202
-----------------------------------
Symbols     : 718
-----------------------------------
Backgrounds : 80
***********************************
